In [1]:
import os
os.chdir("../../")#与easymlops同级目录

### 拆分数据

In [2]:
import pandas as pd
data=pd.read_csv("./data/demo2.csv",encoding="gbk")
data.head(5)

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"


In [3]:
x_train=data[:5000]
x_test=data[5000:]
y_train=x_train["label"]
y_test=x_test["label"]
del x_train["label"]
del x_test["label"]

### pipe构建

In [4]:
from easymlops import NLPPipeline
from easymlops.table.preprocessing import *
from easymlops.nlp.similarity import ElasticSearchSimilarity

In [6]:
nlp=NLPPipeline()
nlp.pipe(FixInput())\
   .pipe(FillNa())\
   .pipe(ElasticSearchSimilarity(col="review",index_name="review_index",es_args=["http://192.168.244.133:9200"],es_kwargs=dict(timeout=60)))

nlp.fit(x_train)

In [7]:
x_test_new=nlp.transform(x_test[:1024])

In [8]:
x_test_new[:5]

,similarity_top_1_raw_data,similarity_top_1_measure,similarity_top_2_raw_data,similarity_top_2_measure,similarity_top_3_raw_data,similarity_top_3_measure
5000,已经第三次入住这家酒店了，除了前台服务很好以外，其他的服务都很好。这次去是免费升级了套房，有...,66.321060,因为未婚夫在美国的关系，他经常要美国、中国两边跑，每次都选择过境香港，所以给了我最好的理由去...,65.817240,我每次定酒店都是在网上翻点评，发现这个酒店的评价不错，而且价格也合理。除了我自己定一间房之外...,63.678680
5001,"酒店还可以,前台的小姐很热情.周边有很多的商店和餐馆.不好的地方是酒店的卡拉ok12店下班,...",38.523030,很喜欢这个酒店，服务特别好，酒店外部环境也很好，在大草坪上放风筝感觉很不错。虽然离市区有点远...,34.792908,"7月25到27,住了三天,从酒店的房间里望外面,能看到海景及红砖的房屋,环境非常美丽,服务也...",32.885834
5002,"由于航班延误,我们错过了机场快线到酒店的穿梭巴士,因此走路冤枉路.从地图上看酒店应该距离湾仔...",107.799550,我和朋友来厦门游玩，听说这家酒店在市政府中心，交通很便利，于是我们选择入住这家酒店。从机场坐...,107.781960,订购马可波罗假日酒店全是看朋友们的点评所致，所以，为了后来的朋友们方便，也要嗦几句！马可波罗...,106.941930
5003,服务很好，尤其是宾馆的客户经理，服务尤其的好，为客人想的很周全，在此点名表扬不太好，房间也不...,22.485111,酒店很超值，干净卫生，尤其值得一提的是在前台办理入住时服务员看到我们携带一个小孩，主动提出酒...,20.930405,去柳州出差不错的选择，尤其是商务房，朝北面的房间夜景不错。宾馆反馈2008年7月31日：非常...,19.987082
5004,每次去杭州入住，有个很重要的原因就是因为四楼餐厅的菜比较好吃，很喜欢。不过，这次有些不快，入...,80.498370,我是7-10号入住的豪华海景大床房，通过携程预定之前，曾给蓝天宾馆打过电话，当时是告知我没房...,79.663680,"第一次入住这家酒店也是携程推荐的,当时还叫台北商务,这次改名红山新世纪了.房间一样价格却是优...",78.804504


### 性能、一致性测试

In [9]:
nlp.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.nlp.similarity.ElasticSearchSimilarity'>) module check [transform] complete,speed:[12.79ms]/it,cpu:[18%],memory:[1328K]

#########################################################################################
 2.空值测试:check_null_value                                                           
 null_values=[None, np.nan, "null", "NULL", "nan", "NaN", "", "none", "None", " "](默认)
########################################################################################
column:[review] check [null value] complete,sp

### 持久化测试

In [10]:
nlp.save("ml.pkl")

In [11]:
nlp=NLPPipeline()
nlp.pipe(FixInput())\
   .pipe(FillNa())\
   .pipe(ElasticSearchSimilarity(col="review",index_name="review_index",es_args=["http://192.168.244.133:9200"],es_kwargs=dict(timeout=60)))

In [12]:
nlp.load("ml.pkl")

In [13]:
nlp.transform_single(x_test[:1].to_dict("records")[0])

{'similarity_top_1_raw_data': '已经第三次入住这家酒店了，除了前台服务很好以外，其他的服务都很好。这次去是免费升级了套房，有厨房，而且有客厅，刚好房间是对着维多利亚港的，能直接看到会展中心，真的感觉很棒！下次会继续入住，并会推荐我的朋友们和客人们入住此间酒店！',
 'similarity_top_1_measure': 66.32106,
 'similarity_top_2_raw_data': '因为未婚夫在美国的关系，他经常要美国、中国两边跑，每次都选择过境香港，所以给了我最好的理由去香港。在香港入住过不同的酒店，这家酒店给我的感觉最好。服务和环境都没有话可说，最感动的是，退房时走得匆忙，一条新买的裙子给忘在酒店房间了，直到回到家收拾行李才发现裙子不见了，当时已经是凌晨2点了，我抱着试试看的心态给酒店去了电话，结果接电话的男士很礼貌的告诉我裙子他们有收到。谢天谢地，经过沟通，以信用卡作邮费支付担保，我的裙子终于在失散一周之后回到了我的身边。下次未婚夫回国，我会强烈推荐他入住这家酒店，我想我以后去香港都会入住这家酒店',
 'similarity_top_2_measure': 65.81724,
 'similarity_top_3_raw_data': '我每次定酒店都是在网上翻点评，发现这个酒店的评价不错，而且价格也合理。除了我自己定一间房之外，给客户也定了一间。到达之后还真是不错，酒店考虑到我和客户接触方面的问题，给我升级房间。头一次遇到升级，开心呢。房间宽敞，干净，服务员热情而且送水果。就是因为地理原因，窗外风景不是很好。但是离万象城比较近，带着客户溜达，他也满高兴的。如果再去深圳，不去香港的话，还是住这里。因为忙，过了20天才来点评。去香港的话，就选择如家渔村小区店，楼下有饭店，水果店，药店，非常方便。就是有点不好找。',
 'similarity_top_3_measure': 63.67868}

In [14]:
nlp.transform_single({"review":"第三次"})

{'similarity_top_1_raw_data': '我们一家三口第二次去三亚，上次住三亚，这次选择亚龙湾。感觉非常好。服务员都很有礼貌、热情。房间舒适，双床房的床基本够大，不用加床了。天天泡在海边，爽！',
 'similarity_top_1_measure': 9.308839,
 'similarity_top_2_raw_data': '这是第三次点评，纯粹是为了拿积分。没什么可以多说了，服务不错。',
 'similarity_top_2_measure': 8.91508,
 'similarity_top_3_raw_data': '是第三次住了，感觉不错，至少是温馨、安全、舒适的，酒店的各场所比较熟，会有主人的感觉。',
 'similarity_top_3_measure': 8.4596}